In [1]:
import psycopg2
import pandas as pd
#pip install psycopg2-binary
#pip install matplotlib
#pip install seaborn
#https://adsai.buas.nl/Year1/BlockD/assets/code_book.html

db_params = {
    'host': '194.171.191.226',
    'port': '6379',
    'database': 'postgres',
    'user': 'group15',
    'password': 'blockd_2024group15_73'
}

con = psycopg2.connect(**db_params)

In [2]:
cursor = con.cursor()
 
cursor.execute("SELECT nspname FROM pg_catalog.pg_namespace;")
rows_psycpopg2 = cursor.fetchall()

In [3]:
# Execute SQL query to fetch distinct category types
query = '''
    SELECT * 
    FROM group15_warehouse.new_accidents_preprocessed
'''
 
cursor.execute(query)
 
# Fetch all rows
new_accident_preprocessed_rows = cursor.fetchall()
 
# Fetch column names
column_names = [desc[0] for desc in cursor.description]
 
# Convert the fetched data into a pandas DataFrame
new_accidents_preprocessed = pd.DataFrame(new_accident_preprocessed_rows, columns=column_names)
 
display(new_accidents_preprocessed)

,Year,Accident severity,Town,First mode of transport,Second mode of transport,Area type,Light condition,Road location,Road condition,Road surface,Road situation,Speed limit,Street,Weather,Accidents
0,2017,Fatal,Breda,Car,Pedestrian,Urban area,Darkness,Intersection,Wet/damp,Brick,Bend,30 km/h,Valkeniersplein,Rain,1
1,2017,Fatal,Breda,Lorry,Other,Urban area,Daylight,Intersection,Wet/damp,Brick,Intersection - 4 arms,50 km/h,Markendaalseweg,Dry,1
2,2017,Fatal,Breda,Lorry,Other,Urban area,Daylight,Road section,Dry,Asphalt (other),Straight road,50 km/h,Academiesingel,Dry,1
3,2017,Injured,Bavel,Car,Lorry,Rural area,Darkness,Road section,Wet/damp,Asphalt (other),Bend,120 km/h,KP ST.ANNABOSCH,Dry,1
4,2017,Injured,Bavel,Car,Other,Rural area,Darkness,Road section,Wet/damp,Porous asphalt,Straight road,130 km/h,RYKSWG,Rain,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6897,2023,Fatal,Breda,Car,Moped,Urban area,Daylight,Road section,Dry,Asphalt (other),Straight road,50 km/h,Terheijdenseweg,Dry,1
6898,2023,Fatal,Breda,Car,Car,Urban area,Darkness,Intersection,Dry,Asphalt (other),Intersection - 4 arms,70 km/h,Rijsbergseweg,Dry,1
6899,2023,Fatal,Breda,Other,Other,Rural area,Daylight,Road section,Wet/damp,Porous asphalt,Straight road,100 km/h,RYKSWG,Dry,1
6900,2023,Fatal,Prinsenbeek,Car,Car,Rural area,Darkness,Road section,Dry,Porous asphalt,Straight road,130 km/h,RYKSWG,Dry,1


In [4]:
import pandas as pd
 
#  Define the updated mapping dictionary
speed_to_road_type = {
    '130 km/h': 'Highway',
    '120 km/h': 'Expressway',
    '100 km/h': 'Rural Highway',
    '90 km/h': 'Major Rural Road',
    '80 km/h': 'Secondary Rural Road',
    '70 km/h': 'Minor Rural Road',
    '50 km/h': 'Urban Road',
    '30 km/h': 'Town Center Road',
    '60 km/h': 'Build-up Road',
    '15 km/h': 'Residential Road',
    'Footpace / homezone': 'Honezone Road',
    'Unknown': 'Unknown'
}
 
#  Create the new column 'Road Type' based on 'Speed limit'
new_accidents_preprocessed['Road type'] = new_accidents_preprocessed['Speed limit'].map(speed_to_road_type)
 
# Check the distribution of the new column
print(new_accidents_preprocessed['Road type'].value_counts())
 
# Display the first few rows to verify the changes
display(new_accidents_preprocessed.head())

Road type
Urban Road              2762
Unknown                 1103
Town Center Road        1058
Highway                  632
Minor Rural Road         535
Rural Highway            359
Expressway               166
Build-up Road            158
Secondary Rural Road      70
Residential Road          31
Major Rural Road          22
Honezone Road              6
Name: count, dtype: int64


,Year,Accident severity,Town,First mode of transport,Second mode of transport,Area type,Light condition,Road location,Road condition,Road surface,Road situation,Speed limit,Street,Weather,Accidents,Road type
0,2017,Fatal,Breda,Car,Pedestrian,Urban area,Darkness,Intersection,Wet/damp,Brick,Bend,30 km/h,Valkeniersplein,Rain,1,Town Center Road
1,2017,Fatal,Breda,Lorry,Other,Urban area,Daylight,Intersection,Wet/damp,Brick,Intersection - 4 arms,50 km/h,Markendaalseweg,Dry,1,Urban Road
2,2017,Fatal,Breda,Lorry,Other,Urban area,Daylight,Road section,Dry,Asphalt (other),Straight road,50 km/h,Academiesingel,Dry,1,Urban Road
3,2017,Injured,Bavel,Car,Lorry,Rural area,Darkness,Road section,Wet/damp,Asphalt (other),Bend,120 km/h,KP ST.ANNABOSCH,Dry,1,Expressway
4,2017,Injured,Bavel,Car,Other,Rural area,Darkness,Road section,Wet/damp,Porous asphalt,Straight road,130 km/h,RYKSWG,Rain,1,Highway


In [5]:
# Label encode categorical variables
from sklearn.preprocessing import LabelEncoder
 
categorical_cols = ['Accident severity', 'Town', 'First mode of transport', 'Second mode of transport',
                    'Area type', 'Light condition', 'Road location', 'Road condition', 'Road surface', 'Road situation', 'Speed limit', 'Street', 'Weather', 'Road type']
 
label_encoders = {}
for column in categorical_cols:
    le = LabelEncoder()
    new_accidents_preprocessed[column] = le.fit_transform(new_accidents_preprocessed[column])
    label_encoders[column] = le  # Store the label encoder for inverse transformation if needed

In [6]:
from sklearn.preprocessing import StandardScaler
 
# Specify numerical features
numerical_cols = ['Year', 'Accidents']
 
# Initialize StandardScaler for numerical data
scaler = StandardScaler()
 
# Scale numerical data
new_accidents_preprocessed[numerical_cols] = scaler.fit_transform(new_accidents_preprocessed[numerical_cols])

In [7]:
from sklearn.model_selection import train_test_split
 
# Separate features and target variable
X = new_accidents_preprocessed.drop(columns=['Accident severity']) 
y = new_accidents_preprocessed['Accident severity']
 
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.3, 
                                                    random_state=42,
                                                    stratify=y)

In [8]:
from imblearn.over_sampling import SMOTE
 
# Apply SMOTE to balance the classes in the training set
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from imblearn.over_sampling import SMOTE
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
 
 
# Define neural network architecture
model = Sequential([
    Dense(256, input_dim=X_train_res.shape[1]),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(128),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(64),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(len(np.unique(y_train)), activation='softmax')  # Output layer with softmax for multi-class classification
])
 
# Compile the model
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
 
# Train the model
history = model.fit(X_train_res, y_train_res, epochs=20, batch_size=32, validation_data=(X_test, y_test))
 
# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy}")
 
# Make predictions
y_pred = np.argmax(model.predict(X_test), axis=-1)
 
# Evaluate predictions
from sklearn.metrics import classification_report, confusion_matrix
 
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=1))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

2024-06-20 11:48:53.363377: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:85: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/20


2024-06-20 11:49:01.179970: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
2024-06-20 11:49:01.180395: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1928] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 45275 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c1:00.0, compute capability: 8.6
I0000 00:00:1718884144.796180 3788684 service.cc:145] XLA service 0x7fd320006310 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1718884144.796244 3788684 service.cc:153]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2024-06-20 11:49:04.872929: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-06-20 11:49:06.597688: I external/local_xla/x

107/389 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3431 - loss: 1.7990 

I0000 00:00:1718884151.417426 3788684 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


389/389 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.3821 - loss: 1.5451 - val_accuracy: 0.5934 - val_loss: 0.8540
Epoch 2/20
389/389 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.4915 - loss: 1.0416 - val_accuracy: 0.4462 - val_loss: 1.0245
Epoch 3/20
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5780 - loss: 0.8997 - val_accuracy: 0.4529 - val_loss: 1.0205
Epoch 4/20
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6112 - loss: 0.8452 - val_accuracy: 0.6369 - val_loss: 0.7402
Epoch 5/20
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6178 - loss: 0.8339 - val_accuracy: 0.4891 - val_loss: 0.9149
Epoch 6/20
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6366 - loss: 0.8056 - val_accuracy: 0.3742 - val_loss: 1.2322
Epoch 7/20
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6356 - loss: 0.8004 - val_accuracy: 0.4911 - val_loss: 0.9807
Epoch 8/20
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6478 - loss: 0.7931 - val_accuracy: 0.5297 - va

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
 
# Define neural network architecture
model = Sequential([
    Dense(512, input_dim=X_train_res.shape[1]),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(256),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(128),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(64),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(len(np.unique(y_train_res)), activation='softmax')  # Output layer with softmax for multi-class classification
])
 
# Compile the model
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
 
# Set up callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
 
# Train the model
history = model.fit(
    X_train_res, y_train_res,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, reduce_lr]
)
 
# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy}")
 
# Make predictions
y_pred = np.argmax(model.predict(X_test), axis=-1)
 
# Evaluate predictions
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=1))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:85: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


389/389 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.3636 - loss: 1.4803 - val_accuracy: 0.5775 - val_loss: 0.9082 - learning_rate: 0.0010
Epoch 2/100
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4678 - loss: 1.0386 - val_accuracy: 0.4925 - val_loss: 0.9174 - learning_rate: 0.0010
Epoch 3/100
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5430 - loss: 0.9332 - val_accuracy: 0.6427 - val_loss: 0.7209 - learning_rate: 0.0010
Epoch 4/100
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5947 - loss: 0.8766 - val_accuracy: 0.5065 - val_loss: 0.8654 - learning_rate: 0.0010
Epoch 5/100
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6228 - loss: 0.8266 - val_accuracy: 0.6055 - val_loss: 0.7039 - learning_rate: 0.0010
Epoch 6/100
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6256 - loss: 0.8180 - val_accuracy: 0.3177 - val_loss: 1.5692 - learning_rate: 0.0010
Epoch 7/100
389/389 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6364 - loss: 0.8017 - val

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
 
 
# Define neural network architecture
model = Sequential([
    Dense(512, kernel_regularizer=l2(0.01), input_dim=X_train_res.shape[1]),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(256, kernel_regularizer=l2(0.01)),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(128, kernel_regularizer=l2(0.01)),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(64, kernel_regularizer=l2(0.01)),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(),
    Dropout(0.5),
 
    Dense(len(np.unique(y_train_res)), activation='softmax')
])
 
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
 
# Set up callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
 
# Train the model
history = model.fit(
    X_train_res, y_train_res,
    epochs=100,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, reduce_lr]
)
 
# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy}")
 
# Make predictions
y_pred = np.argmax(model.predict(X_test), axis=-1)
 
# Evaluate predictions
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=1))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:85: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


195/195 ━━━━━━━━━━━━━━━━━━━━ 17s 38ms/step - accuracy: 0.3714 - loss: 7.1688 - val_accuracy: 0.6171 - val_loss: 4.4893 - learning_rate: 0.0010
Epoch 2/100
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4274 - loss: 4.1977 - val_accuracy: 0.4351 - val_loss: 2.8635 - learning_rate: 0.0010
Epoch 3/100
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5058 - loss: 2.5670 - val_accuracy: 0.4930 - val_loss: 1.8540 - learning_rate: 0.0010
Epoch 4/100
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5280 - loss: 1.7778 - val_accuracy: 0.7335 - val_loss: 1.1186 - learning_rate: 0.0010
Epoch 5/100
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5524 - loss: 1.4001 - val_accuracy: 0.6335 - val_loss: 1.0682 - learning_rate: 0.0010
Epoch 6/100
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5783 - loss: 1.2003 - val_accuracy: 0.1449 - val_loss: 1.8404 - learning_rate: 0.0010
Epoch 7/100
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5667 - loss: 1.1545 - val